In [ ]:
# !cat -n googleplaystore.csv | tr -s '\r\n' '\n' | grep -C2 'Life Made WI-Fi Touchscreen'

In [ ]:
# filter = ( goo["App"] == "ROBLOX" )
# goo[filter]


# Advanced Pandas

This lecture covers an array of advanced pandas methods that may be useful as you continue to work on more challenging data science problems. Topics that are covered include:
- Accessor Methods
- Filtering Data
- Changing Data Types
- Apply, Group By
- Merging & Reshaping DataFrames

## Read in Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Read in the googleplaystore.csv data
url = "http://ddc-datascience.s3.amazonaws.com/googleplaystore.csv"
url

In [ ]:
!curl -s -I -XHEAD {url}


In [ ]:
!ls -la

In [ ]:
!curl -s -O {url}


In [ ]:
url

In [ ]:
file_csv = url.split("/")[-1]
file_csv

In [ ]:
!ls -la {file_csv}

In [ ]:
goo = pd.read_csv( file_csv )
goo.head()

In [ ]:
goo.shape

In [ ]:
goo.info()

In [ ]:
# Make a copy for cleaning
google = goo.copy()

In [ ]:
# Take a look at the data types
google.dtypes

In [ ]:
google.select_dtypes(include = ['float64', 'int64'])


## Filtering

In [ ]:
# Look at values in the 'Installs' column
google['Installs'].value_counts()

In [ ]:
filter = ( goo["Installs"] == "Free" )
goo[filter]


In [ ]:
filter.shape

In [ ]:
goo.iloc[[10472]]

In [ ]:
google.iloc[[10472]]

In [ ]:
# Let's remove the 'Free' row
# Find row
np.where(google['Installs'] == 'Free')


In [ ]:
# Compare with "0+"
# Find row
np.where(google['Installs'] == '0+')[0]


In [ ]:
# Let's remove the 'Free' row
# Get index value
np.where(goo['Installs'] == 'Free')[0]

In [ ]:
# Drop that row
index_filter = ( np.where(goo['Installs'] == 'Free')[0] )
google.drop(index_filter, inplace = True)

In [ ]:
# Confirm it dropped by looking at value counts of Installs column
google['Installs'].value_counts()

## Accessor Methods

Accessors are used to access methods that correspond to certain data types. We are going to focus on two commonly used accessors:
- `str` (string)
- `dt` (datetime)



### str

In [ ]:
google['Price'].info()

In [ ]:
# Look at the value counts of the Price column
google['Price'].value_counts()

In [ ]:
# Let's remove the $ by replacing $ with a blank string
google['Price_new1'] = google['Price'].str.replace('$', '', regex = False)

In [ ]:
google

In [ ]:
# Confirm it worked by looking at the first five rows of the Price column
google[['Price','Price_new1']].value_counts()

In [ ]:
google.assign(Price = google["Price_new1"]).drop(columns = ['Price_new1'])

In [ ]:
google = google.drop(columns = ['Price']).rename( columns = { 'Price_new1' : 'Price' })
google

In [ ]:
goo.dtypes

In [ ]:
google.info()

In [ ]:
google['Installs'].value_counts()

In [ ]:
# Remove the + and , from Installs column. Confirm it worked by looking at the first five rows.
google['Installs'] = google['Installs'].str.replace(',','', regex = False)
google['Installs'] = google['Installs'].str.replace('+','', regex = False)
google['Installs'].head()

In [ ]:
goo.dtypes

In [ ]:
google.dtypes

In [ ]:
# Look at the first five rows of the Type column
google['Type'].head()

In [ ]:
# Look at the first five rows of the Type column
google['Type'].value_counts()

In [ ]:
# Make the Type column lowercase
google['Type'] = google['Type'].str.lower()
google['Type'].head()

In [ ]:
# Look at the first five rows of the Type column
google['Type'].value_counts()

### dt

In [ ]:
google.dtypes.sort_values()

In [ ]:
# Look at the Last Updated column
google['Last Updated'].head()

In [ ]:
# Change this column to datetime
google['Last Updated'] = pd.to_datetime(google['Last Updated'])
google['Last Updated'].head()

In [ ]:
google.dtypes

In [ ]:
# See how many days are in each month
google['Last Updated'].dt.days_in_month

In [ ]:
# Extract the year from the datetime object
google['Last Updated'].dt.year

In [ ]:
# Extract the month from the datetime object
google['Last Updated'].dt.month

In [ ]:
google['Last Updated'].dt.quarter

In [ ]:
# Extract the month from the datetime object
google['Last Updated'].dt.day_name().value_counts()

In [ ]:
google.info()

## Data Types

There are two ways to convert data types:
1. `.astype()` - typically used when the conversion is straightforward
2. `pd.to_numeric()` or `pd.to_datetime()` - has options for you to specify what to do if there is an error in the conversion

In [ ]:
# Look at data types
google.dtypes

In [ ]:
google['Installs']

In [ ]:
# Change Installs column to be integer
google['Installs'] = google['Installs'].astype('int')
google['Installs'].head()

In [ ]:
# Change Price column to be float
google['Price'] = google['Price'].astype('float')
google['Price'].head()

In [ ]:
# Look at data types
google.dtypes

## Apply

In [ ]:
# See current data types
google.dtypes

In [ ]:
# Separate numeric data
num_google = google.select_dtypes(include = ['float64', 'int64'])
num_google.head()

In [ ]:
num_google.dtypes

In [ ]:
# Calculate the mean of each column using apply()
num_google.apply(np.mean)

In [ ]:
num_google.apply(sum)

In [ ]:
num_google.info()

In [ ]:
num_google.apply(np.sum)

In [ ]:
num_google.apply(pd.notnull).sum()

In [ ]:
# Calculate the maximum of each column using apply()
num_google.apply(np.max)

## Group By

In [ ]:
google.head()

In [ ]:
google["Category"].value_counts()

In [ ]:
# Calculate average rating by category
google.groupby(['Category'])['Category'].apply(len).sort_values( ascending=False )

In [ ]:
google['Rating'].mean()

In [ ]:
# Calculate average rating by category
google.groupby('Category')['Rating'].mean()

In [ ]:
# Let's make a plot!
ratings_by_cat = google.groupby('Category')['Rating'].mean().sort_values( ascending=True)
ratings_by_cat.plot(kind = 'barh')
plt.ylabel('Rating') ;

In [ ]:
# Calculate median installs by type
google.groupby('Type')['Installs'].median()

## Merging Data Frames

In [ ]:
# Read in googleplaystore_user_reviews.csv
url = "http://ddc-datascience.s3.amazonaws.com/googleplaystore_user_reviews.csv"
rev = pd.read_csv(url)
rev.head()

In [ ]:
rev.shape

In [ ]:
# Look at google dataframe
google.head()

In [ ]:
google["App"].value_counts().head()

In [ ]:
rev["App"].value_counts().head()

In [ ]:
# Combine dataframes using pd.merge.
# Note: there are a lot of options you can change in pd.merge()
google_and_rev = pd.merge(google, rev, on = 'App')
google_and_rev.head()

In [ ]:
google.shape

In [ ]:
(google['App'] == "CBS Sports App - Scores, News, Stats & Watch Live").sum()

In [ ]:
rev.shape

In [ ]:
(rev['App'] == "CBS Sports App - Scores, News, Stats & Watch Live").sum()

In [ ]:
google_and_rev.shape

In [ ]:
(google_and_rev['App'] == "CBS Sports App - Scores, News, Stats & Watch Live").sum()

In [ ]:
google_and_rev["App"].value_counts().head()

In [ ]:
8*320

In [ ]:
data1 = {'ID': [1, 2, 3]}
df1 = pd.DataFrame(data1)
df1


In [ ]:
# Create the second DataFrame
data2 = {'Name': ['A', 'B', 'C']}
df2 = pd.DataFrame(data2)
df2


In [ ]:
# Perform a cross join
cross_join = df1.assign(key=1).merge(df2.assign(key=1), on='key').drop('key', axis=1)
cross_join


In [ ]:
cross_join = cross_joined_df = pd.merge(df1, df2, how='cross')
cross_join

## Reshaping DataFrames

In [ ]:
# Extract the first 10 rows of the google_and_rev dataframe
first_10 = google_and_rev.iloc[:10].copy()
first_10

In [ ]:
# Transpose the data set
first_10_t = first_10.T
first_10_t

In [ ]:
google.describe().transpose()

In [ ]:
google.query?

In [ ]:
(
  google
    .describe()
    .transpose()
    .query("count < 10_000")
)

In [ ]:
(
  google
    .describe()
    .transpose()
    .info()
)

In [ ]:
# Detect positive skew
pd.set_option('display.precision', 2)
(    # parentheses needed for method chaining
google
  .describe()
  .transpose()
  .astype({
      "count": int,
  })
  .rename({"50%":"median"}, axis = 1)
  .query("mean > median")
)

In [ ]:
(
    1
  + 2
    )

In [ ]:
( google.describe().transpose() )

Other methods:
- pivot()
- pivot_table()
- melt()
- stack()
- unstack()
- explode()
- get_dummies()
- crosstab()

References

https://towardsdatascience.com/learn-advanced-features-for-pythons-main-data-analysis-library-in-20-minutes-d0eedd90d086
https://medium.com/@tamoghnasaha.22/all-you-really-need-to-know-python-notebook-advanced-pandas-da1697aee647

## Pandas and Numpy introspection


In [ ]:
len([ x for x in dir(pd.DataFrame) if not x.startswith("_") ])

In [ ]:
len([ x for x in dir(pd.Series) if not x.startswith("_") ])

In [ ]:
len([ x for x in dir(np.ndarray) if not x.startswith("_") ])